# RTS fusion experiments E1-E4 (Kaggle driver)
Thin driver for the code in `Experiments/`. Upload that folder (including `splits/grouped_folds.json`) as a Kaggle dataset, and add the competition release as a second dataset.
**Edit only the first cell.** Every run appends one row to `OUT_DIR/results/results.csv`; the last cell prints mean ± std across folds.

`FOLDS_FILE` is the leak-free grouped split (`make_grouped_split.py`): overlapping chips never straddle train/val/test, and its `test_ids` are held out of every fold. After each run the best checkpoint is also scored on that local test set (`results_test.csv`), which is the number to compare with the leaderboard.

One run (experiment × fold × seed, 50 epochs, fp32) is about 85–90 min on a T4 (~103 s/epoch for E1), so 20 runs need about 3 sessions of 12 h. Split them with `EXPERIMENTS` / `FOLDS`, and list earlier sessions' `results.csv` in `PREVIOUS_RESULTS`.

In [ ]:
# ============================== constants: edit only this cell ==============================
DATA_ROOT   = "/kaggle/input/rts-challenge/competition_release"  # release: train/, metadata/, tools/
CODE_ROOT   = "/kaggle/input/rts-fusion-code/Experiments"        # this repo's Experiments/ folder
OUT_DIR     = "/kaggle/working/outputs"                          # checkpoints, TensorBoard, results.csv
CACHE_DIR   = "/kaggle/working/cache"                            # NaN-filled tiles (~1.2 GB) + fold stats
FOLDS_FILE  = "splits/grouped_folds.json"   # grouped CV + held-out local test; "splits/folds.json" = old leaky random folds

EXPERIMENTS = ["e1", "e2", "e3", "e4"]
FOLDS       = [0, 1, 2, 3, 4]
SEEDS       = [0]

EPOCHS      = 50
BATCH_SIZE  = 8
NUM_WORKERS = 4
AMP         = False           # fp16 autocast makes Swin-V2 produce NaN / skipped steps; keep fp32
PRETRAINED  = True            # Swin_V2_B_Weights.NAIP_RGB_SI_SATLAS (needs internet)

DROP_NDWI        = False      # E2 ablation: 7 channels
NDWI_TO_TERRAIN  = False      # E3/E4 ablation: NDWI as a 4th terrain channel
MASK_LOGITS_INIT_STD = 0.001  # None = torchvision default init (loss_mask ~9 at start); 0.001 = detectron2-style

RUN_SMOKE_TEST   = True
PREVIOUS_RESULTS = []         # results.csv files from earlier sessions, merged into the summary

In [ ]:
import os, subprocess, sys
!pip install -q torchgeo timm
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # CODE_ROOT is read-only
sys.path.insert(0, CODE_ROOT)

def overrides(exp):
    o = [f"data.root={DATA_ROOT}", f"data.cache_dir={CACHE_DIR}", f"out_dir={OUT_DIR}",
         f"data.folds_file={FOLDS_FILE}",
         f"train.epochs={EPOCHS}", f"train.batch_size={BATCH_SIZE}", f"data.num_workers={NUM_WORKERS}",
         f"train.amp={str(AMP).lower()}", f"model.pretrained={str(PRETRAINED).lower()}",
         f"model.mask_logits_init_std={'null' if MASK_LOGITS_INIT_STD is None else MASK_LOGITS_INIT_STD}"]
    if exp == "e2":
        o.append(f"flags.drop_ndwi={str(DROP_NDWI).lower()}")
    if exp in ("e3", "e4"):
        o.append(f"flags.ndwi_to_terrain={str(NDWI_TO_TERRAIN).lower()}")
    return o

def run(script, *args):
    """Run a script from CODE_ROOT and stream its output into the cell."""
    cmd = [sys.executable, f"{CODE_ROOT}/{script}", *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail = (tail + [line])[-30:]
    if p.wait():
        raise RuntimeError(f"{script} exited with {p.returncode}. Last output: " + "".join(tail))

In [ ]:
# Smoke test: random 8x333x417 batch through every experiment (train + eval mode).
if RUN_SMOKE_TEST:
    run("smoke_test.py", *(["--pretrained"] if PRETRAINED else []), "--override", f"data.root={DATA_ROOT}")

In [ ]:
for exp in EXPERIMENTS:
    for fold in FOLDS:
        for seed in SEEDS:
            run("train.py", "--config", f"{CODE_ROOT}/configs/{exp}.yaml", "--fold", fold, "--seed", seed,
                "--override", *overrides(exp))
            if "grouped" in FOLDS_FILE:   # best checkpoint on the held-out local test chips
                run("eval.py", "--config", f"{CODE_ROOT}/configs/{exp}.yaml", "--fold", fold, "--seed", seed,
                    "--split", "test", "--override", *overrides(exp))

In [ ]:
import pandas as pd
for name, split in [("results.csv", "val"), ("results_test.csv", "test")]:
    res = f"{OUT_DIR}/results/{name}"
    frames = [pd.read_csv(p) for p in (os.path.join(os.path.dirname(q), name) for q in PREVIOUS_RESULTS)
              if os.path.exists(p)]
    if frames:
        if os.path.exists(res):
            frames.append(pd.read_csv(res))
        os.makedirs(os.path.dirname(res), exist_ok=True)
        pd.concat(frames).to_csv(res, index=False)
    if os.path.exists(res):
        print(f"===== {split} =====")
        run("eval.py", "--summary", "--split", split, "--out_dir", OUT_DIR)
        display(pd.read_csv(res))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT_DIR}/runs